# Spherical Neural Surface (SNS) — Interactive Demo

A **Spherical Neural Surface (SNS)** is a neural network $f : (S² \in ℝ³) → ℝ³$ that parametrises a 3D surface using the unit sphere as domain. Because the map is differentiable everywhere, we can compute surface normals, curvatures, and other geometric quantities in a continuous way (using auto-diff) — no discrete differential geometry required.

**What this notebook does:**
1. Loads a pretrained SNS model
2. Evaluates the map on a subdivided icosphere (the parametrisation domain)
3. Computes the chosen geometric quantity via automatic differentiation (`BatchDiffQuant`)
4. Opens an interactive Open3D viewer showing the result on the surface and on the sphere

**Suggestions for exploring the code:**
1. Try visualising various differential quantities and scalarfields (edit the Configuration cell - scroll down)
2. Have a look at the computations in `sns/neural_surfaces-main/differential/differential.py` to see how we compute First Fundamental Form, Second Fundamental Form, derived quantities and so on.
3. You could try implementing some more differential quantities e.g. a conformal energy or an isometric energy. For example, several types of distortion energy are categorised in a very useful table: Table II of [Scalable Locally Injective Mappings [Rabinovich et al. 2017]](https://dl.acm.org/doi/10.1145/2983621)

---
## Required files

Make sure the following exist **relative to this notebook** (i.e., inside the `sns/` directory) before running:

| Path | Purpose |
|---|---|
| `neural_surfaces-main/` | Core library — models, differential geometry |
| `demo_utils.py` | Visualisation helpers (lives next to this notebook) |
| `data/analytic/sphere/sphere{0–7}.obj` | Icosphere domain meshes |
| `data/SNS/{SURF_NAME}/weights.pth` | Pretrained SNS model weights |
| `data/eigenfunc/{SURF_NAME}/orthoweights/` | *(Only for `neural_eigenfunctions` mode)* |

**Available pretrained models** (set as `SURF_NAME` in the Configuration cell):
- `HUMAN24461`, available with 5 optimised eigenfunctions
- `MAX10606`
- `ARMADILLO21622`
- `SMALLFLOWER`

---

## Setup

Run this cell once at the start — it adds the library to the module search path and imports everything the notebook needs.

In [14]:
import sys, os

import numpy as np
import torch

import trimesh
import matplotlib.pyplot as plt


# Locate neural_surfaces-main robustly — safe to re-run.
_cwd = os.path.abspath('.')
if os.path.isdir(os.path.join(_cwd, 'neural_surfaces-main')):
    _here, NS_ROOT = _cwd, os.path.join(_cwd, 'neural_surfaces-main')
elif os.path.basename(_cwd) == 'neural_surfaces-main':
    NS_ROOT, _here = _cwd, os.path.dirname(_cwd)
else:
    raise RuntimeError(
        f"Cannot locate neural_surfaces-main from {_cwd}.\n"
        "Please launch Jupyter from the sns/ directory."
    )
for path in (NS_ROOT, _here):
    if path not in sys.path:
        sys.path.insert(0, path)
os.chdir(NS_ROOT)


from models.mlp import ResidualMLP
from differential import DifferentialModule, BatchDiffQuant
from demo_utils import (
    launch_viewer, launch_colorbar, o3d_mesh, arrow_mesh,
    scalar_to_rgb,
)

print('Setup complete. Working directory:', os.getcwd())

Setup complete. Working directory: /Users/rubato/Desktop/Bonn/Bonn-Mathematical-Surface-Processing-Summer-School-2026/sns/neural_surfaces-main


---
## Configuration

**This is the main cell to edit.** Change `SURF_NAME` and `VIS_MODE` to explore different surfaces and visualisations. You can usually leave everything else at its default.

In [15]:
# ── Surface ───────────────────────────────────────────────────────────────────
# Options: 'HUMAN24461' | 'MAX10606' | 'ARMADILLO21622' | 'SMALLFLOWER'
SURF_NAME = 'SMALLFLOWER'

# ── Icosphere resolution ──────────────────────────────────────────────────────
# Level 3 ≈ 10 k vertices (faster preview)
# Level 7 ≈ 164 k vertices (high detail)

#Level 6 or 7 with SECOND ORDER differential quantities might take too long on some machines.
# On my CPU, level 7 with mean curvature and batch-size 10000 takes about 90 seconds, level 6 takes about 25 seconds.
# FIRST ORDER differential quantities (normals, and things using only first fundamental form) are much quicker.
SPHERE_LEVEL = 6

# ── Memory budget ────────────────────────────────────────────────────────────
BATCH_SIZE = 10_000   # reduce if you run out of memory

# ── Visualisation mode ────────────────────────────────────────────────────────
# 'normals'              — surface normals encoded as RGB #FIRST ORDER
# 'area_distortion'      — local area ratio relative to the sphere #FIRST ORDER

# 'mean_curvature'       — mean curvature H #SECOND ORDER
# 'gaussian_curvature'   — Gaussian curvature K #SECOND ORDER
# 'max_curvature'        — maximum absolute principal curvature #SECOND ORDER
# 'principal_directions' — crossfield: ±dir₁ (blue) and ±dir₂ (red) #SECOND ORDER
# 'laplace_beltrami'     — Laplace–Beltrami of a sine test field #SECOND ORDER (uses mean curvature)

# 'neural_eigenfunctions'    — Laplace Beltrami eigenfunctions from data/eigenfunc/{SURF_NAME}/ #NO DIFFERENTIATION - just evaluates scalarfield models
VIS_MODE = 'normals'




if not VIS_MODE in ['normals', 'area_distortion', 'gaussian_curvature', 'max_curvature', 'mean_curvature',
                    'principal_directions', 'laplace_beltrami', 'neural_eigenfunctions']:
    raise ValueError('Sorry,', VIS_MODE, 'is not one of the implemented visualisation modes.')


if SPHERE_LEVEL>=6 and VIS_MODE in ['gaussian_curvature', 'max_curvature', 'mean_curvature',
                    'principal_directions', 'laplace_beltrami']:
    print('Warning: high resolution computation of second order differential quantities might be a bit slow.')

In [16]:
# ── Specific visualisation settings — safe to leave as defaults ─────────────────────────────

# Colormap and scaling are chosen automatically from VIS_MODE;
# override here only if you want a different look.
_auto_cmap = {
    'mean_curvature': 'seismic', 'gaussian_curvature': 'seismic',
    'max_curvature':  'seismic', 'laplace_beltrami':   'viridis',
    'area_distortion': 'hot',
}
_auto_scaling = {
    'mean_curvature': 'linear9',  'gaussian_curvature': 'linear5',
    'max_curvature':  'mapping12', 'area_distortion':   'logmap',
    'laplace_beltrami': 'linear5',
}
CMAP    = _auto_cmap.get(VIS_MODE)
SCALING = _auto_scaling.get(VIS_MODE)

# Principal-direction crossfield
PCD_ARROW_LENGTH  = 0.005
PCD_OFFSET_FACTOR = 0.01
PCD_RATIO         = 10.0
PCD_SUBSAMPLE     = 5

# Eigenfunction viewer
EIGENFUNC_INDICES = None     # None = all; or e.g. [0, 1, 2] to pick specific ones
EIGENFUNC_SCALE   = 5.0
EIGENFUNC_CMAP    = 'Spectral'

# Laplace–Beltrami test field
LB_FUNC_FREQ  = 15.0
LB_FUNC_SCALE = 3.0

---
## Step 1 — Load pretrained SNS model

The SNS model is a residual MLP that maps sphere points (x, y, z) ∈ S² to surface points in ℝ³. The architecture below matches the weights in `data/SNS/{SURF_NAME}/weights.pth`.

In [17]:
MODEL_CFG = {
    'input_size':  3,
    'output_size': 3,
    'layers':      [256, 256, 256, 256, 256, 256, 256, 256],
    'act':         'Softplus',
    'act_params':  {},
    'bias':        False,
    'norm':        None,
    'drop':        0.0,
}

model = ResidualMLP(MODEL_CFG)
weights_path = os.path.join('..', 'data', 'SNS', SURF_NAME, 'weights.pth')
model.load_state_dict(torch.load(weights_path, map_location='cpu'))
model.eval()
print(f'Loaded SNS model for {SURF_NAME}')

dict_keys(['input_size', 'output_size', 'layers', 'act', 'act_params', 'bias', 'norm', 'drop'])
Res MLP init map weights called
yh 1
Loaded SNS model for SMALLFLOWER


---
## Step 2 — Load sphere mesh (parametrisation domain)

The unit sphere S² is the domain of the SNS map. We sample it using a subdivided icosphere — a mesh with nearly-uniform triangles. Higher levels give more vertices and a finer approximation.

In [18]:
import glob

sphere_dir  = os.path.join('..', 'data', 'analytic', 'sphere')
sphere_path = os.path.join(sphere_dir, f'sphere{SPHERE_LEVEL}.obj')

if not os.path.exists(sphere_path):
    available = sorted(
        os.path.basename(f) for f in glob.glob(os.path.join(sphere_dir, 'sphere*.obj'))
    )
    available_str = ', '.join(available) if available else 'none found'
    raise FileNotFoundError(
        f"Icosphere level {SPHERE_LEVEL} not found at {sphere_path}\n"
        f"Available levels in that folder: {available_str}\n"
        f"Add more sphere{{N}}.obj files (N = 0-7) to {sphere_dir} to use higher levels."
    )

tm_sphere = trimesh.load(sphere_path)
vertices  = np.array(tm_sphere.vertices, dtype=np.float32)
faces     = np.array(tm_sphere.faces)
print(f'Sphere level {SPHERE_LEVEL}: {vertices.shape[0]:,} vertices, {faces.shape[0]:,} faces')


Sphere level 6: 40,962 vertices, 81,920 faces


---
## Step 3 — Compute

`BatchDiffQuant` pushes the sphere vertices through the SNS model in batches, using automatic differentiation to compute the Jacobian and from it the geometric quantities (normals, curvatures, etc.).

In [19]:
diffmod = DifferentialModule()
bdq     = BatchDiffQuant(vertices, model, diffmod, batch_size=BATCH_SIZE)

if VIS_MODE == 'normals':
    bdq.compute_normals()
elif VIS_MODE in ('mean_curvature', 'gaussian_curvature'):
    bdq.compute_curvature()
elif VIS_MODE in ('max_curvature', 'principal_directions'):
    bdq.compute_directions()
elif VIS_MODE == 'area_distortion':
    bdq.compute_area_distortions()
elif VIS_MODE == 'laplace_beltrami':
    bdq.compute_curvature()
    bdq.compute_normals()
elif VIS_MODE == 'neural_eigenfunctions':
    bdq.compute_SNS()

all_output_vertices = bdq.all_output_vertices
print(f'Done — {all_output_vertices.shape[0]:,} surface vertices computed (mode: {VIS_MODE!r})')

Done — 40,962 surface vertices computed (mode: 'normals')


In [20]:
# ── Laplace–Beltrami of a sine test field (only runs when VIS_MODE == 'laplace_beltrami') ──
#
# Uses the formula: LB(f) = Δ(f) − 2H(∇f · n) − nᵀ Hess(f) n
# where H and n come from the SNS model, and the analytic derivatives of f
# are supplied explicitly.

if VIS_MODE == 'laplace_beltrami':
    from torch.nn import functional as F

    # f(x) = scale * sin(freq * (x · d))  for a fixed random unit direction d
    torch.manual_seed(1)
    d    = F.normalize(torch.randn(1, 3), dim=1).squeeze()
    surf = torch.tensor(all_output_vertices, dtype=torch.float32)
    arg  = LB_FUNC_FREQ * (surf * d).sum(-1)

    func_vals = LB_FUNC_SCALE * torch.sin(arg)
    grad_vals = LB_FUNC_SCALE * d * (LB_FUNC_FREQ * torch.cos(arg)).unsqueeze(-1)
    hess_vals = (
        (d.unsqueeze(-1) @ d.unsqueeze(0)).unsqueeze(0)
        * (LB_FUNC_SCALE * (-LB_FUNC_FREQ**2) * torch.sin(arg))
          .unsqueeze(-1).unsqueeze(-1)
    )

    lb_values = diffmod.laplace_beltrami_MC(
        normals   = torch.tensor(bdq.all_normals, dtype=torch.float32),
        meancurv  = torch.tensor(bdq.all_H,       dtype=torch.float32),
        f         = func_vals,
        grad_f    = grad_vals,
        hessian_f = hess_vals,
    ).detach().numpy()

    print(f'f(x):  [{func_vals.min():.3f}, {func_vals.max():.3f}]')
    print(f'LB(f): [{lb_values.min():.3f}, {lb_values.max():.3f}]')
else:
    print(f'Laplace–Beltrami cell skipped (VIS_MODE={VIS_MODE!r})')

Laplace–Beltrami cell skipped (VIS_MODE='normals')


---
## Step 4 — Map values to vertex colours

In [21]:
vertex_rgb = None

if VIS_MODE == 'normals':
    vertex_rgb = np.clip(0.5 * bdq.all_normals + 0.5, 0.0, 1.0)
elif VIS_MODE == 'mean_curvature':
    vertex_rgb = scalar_to_rgb(bdq.all_H,                 CMAP, SCALING)
elif VIS_MODE == 'gaussian_curvature':
    vertex_rgb = scalar_to_rgb(bdq.all_K,                 CMAP, SCALING)
elif VIS_MODE == 'max_curvature':
    vertex_rgb = scalar_to_rgb(bdq.all_MAC,               CMAP, SCALING)
elif VIS_MODE == 'area_distortion':
    vertex_rgb = scalar_to_rgb(bdq.all_area_distortions,  CMAP, SCALING)
elif VIS_MODE == 'laplace_beltrami':
    func_vertex_rgb = scalar_to_rgb(func_vals.numpy(), CMAP, 'linear10')
    vertex_rgb      = scalar_to_rgb(lb_values,          CMAP, 'linear500')

if vertex_rgb is not None:
    print(f'vertex_rgb: {vertex_rgb.shape},  range [{vertex_rgb.min():.3f}, {vertex_rgb.max():.3f}]')
elif VIS_MODE == 'principal_directions':
    print('principal_directions — crossfield will be built in the viewer cell')
elif VIS_MODE == 'neural_eigenfunctions':
    print('neural_eigenfunctions — see the Eigenfunction section below')

vertex_rgb: (40962, 3),  range [0.000, 1.000]


In [22]:
if VIS_MODE not in ('normals', 'principal_directions', 'neural_eigenfunctions'):
    if VIS_MODE == 'laplace_beltrami':
        entries = [('f(x)', 'linear10'), ('LB(f)', 'linear500')]
    else:
        entries = [(VIS_MODE.replace('_', ' '), SCALING)]
    launch_colorbar(entries, CMAP, title=VIS_MODE.replace('_', ' '))
    print(f'Colorbar window launched for {VIS_MODE!r}')
else:
    print(f'No colorbar for {VIS_MODE!r} mode.')


No colorbar for 'normals' mode.


---
## Step 5 — View in Open3D

Interactive windows open in a separate process — you can keep them open while the notebook continues running. Close a window to free it.

In [23]:
# ── Surface viewer ────────────────────────────────────────────────────────────
if VIS_MODE == 'principal_directions':
    step = PCD_SUBSAMPLE
    pts  = bdq.all_output_vertices[::step]
    d1   = bdq.all_directions[0][::step]
    d2   = bdq.all_directions[1][::step]
    nrm  = bdq.all_normals[::step]
    al, off, rat = PCD_ARROW_LENGTH, PCD_OFFSET_FACTOR, PCD_RATIO

    Va, Fa = arrow_mesh(pts,  d1, d2, nrm, al, off, rat)
    Vc, Fc = arrow_mesh(pts, -d1, d2, nrm, al, off, rat)
    V_blue = np.vstack([Va, Vc])
    F_blue = np.vstack([Fa, Fc + len(Va)])

    Vb, Fb = arrow_mesh(pts,  d2, d1, nrm, al, off, rat)
    Vd, Fd = arrow_mesh(pts, -d2, d1, nrm, al, off, rat)
    V_red  = np.vstack([Vb, Vd])
    F_red  = np.vstack([Fb, Fd + len(Vb)])

    grey = np.full((len(bdq.all_output_vertices), 3), 0.75)
    launch_viewer(
        o3d_mesh(bdq.all_output_vertices, faces, grey),
        o3d_mesh(V_blue, F_blue, np.tile([0.0, 0.2, 0.9], (len(V_blue), 1))),
        o3d_mesh(V_red,  F_red,  np.tile([0.9, 0.1, 0.1], (len(V_red),  1))),
        window_name=f'Principal curvature directions — {SURF_NAME}',
    )
    print('Crossfield viewer launched  (blue = dir₁, red = dir₂)')

elif VIS_MODE != 'neural_eigenfunctions':
    launch_viewer(
        o3d_mesh(all_output_vertices, faces, vertex_rgb),
        window_name=f'SNS surface ({SURF_NAME}) — {VIS_MODE}',
    )
    print(f'Surface viewer launched')

    if VIS_MODE == 'laplace_beltrami':
        launch_viewer(
            o3d_mesh(all_output_vertices, faces, func_vertex_rgb),
            window_name=f'SNS surface ({SURF_NAME}) — scalar field f(x)',
        )
        print('Scalar field f(x) viewer also launched')

Surface viewer launched


In [24]:
# ── Sphere viewer — same colours mapped back onto the domain ─────────────────
# Useful for understanding how the SNS map distorts the sphere.
if VIS_MODE not in ('principal_directions', 'neural_eigenfunctions'):
    launch_viewer(
        o3d_mesh(vertices, faces, vertex_rgb),
        window_name=f'Sphere domain — {SURF_NAME} — {VIS_MODE}',
    )
    print('Sphere viewer launched — same colours as the surface above')

Sphere viewer launched — same colours as the surface above


---
## Eigenfunction viewer *(neural_eigenfunctions mode only)*

When `VIS_MODE = 'neural_eigenfunctions'`, this section loads pretrained eigenfunctions from `data/eigenfunc/{SURF_NAME}/orthoweights/` and opens a viewer for each one on both the surface and the sphere.

Settings are controlled by `EIGENFUNC_INDICES`, `EIGENFUNC_SCALE`, and `EIGENFUNC_CMAP` in the Advanced configuration cell.

In [25]:
import json, glob

if VIS_MODE == 'neural_eigenfunctions':
    # Load the eigenfunc model architecture from a JSON config.
    ef_cfg_path = os.path.join('experiment_configs', 'eigenfunc', f'{SURF_NAME}.json')
    if not os.path.exists(ef_cfg_path):
        ef_cfg_path = os.path.join('experiment_configs', 'eigenfunc', 'GENERIC.json')
    with open(ef_cfg_path) as fj:
        ef_cfg = json.load(fj)
    ef_model = ResidualMLP(ef_cfg['models']['structure'])
    ef_model.eval()
    print(f'Eigenfunc model loaded (config: {os.path.basename(ef_cfg_path)})')

    # Find orthoweights.
    weights_dir  = os.path.join('..', 'data', 'eigenfunc', SURF_NAME, 'orthoweights')
    weight_files = sorted(glob.glob(os.path.join(weights_dir, 'ortho*.pth')))
    if not weight_files:
        print(f'No orthoweights found at {weights_dir}')
    else:
        print(f'Found {len(weight_files)} eigenfunction(s)')
        indices  = list(range(len(weight_files))) if EIGENFUNC_INDICES is None else EIGENFUNC_INDICES
        cmap_ef  = plt.get_cmap(EIGENFUNC_CMAP)
        EF_BATCH = 2000

        for idx in indices:
            if idx >= len(weight_files):
                print(f'  Index {idx} out of range ({len(weight_files)} available), skipping')
                continue

            ef_name = os.path.basename(weight_files[idx]).replace('.pth', '')
            ef_model.load_state_dict(torch.load(weight_files[idx], map_location='cpu'))
            ef_model.eval()

            # Run inference on sphere vertices in small batches.
            n       = vertices.shape[0]
            ef_vals = np.zeros(n)
            for i in range(0, n, EF_BATCH):
                batch = torch.tensor(vertices[i:i+EF_BATCH], dtype=torch.float32)
                with torch.no_grad():
                    ef_vals[i:i+EF_BATCH] = ef_model(batch).mean(-1).numpy()

            ef_rgb = cmap_ef(np.clip(EIGENFUNC_SCALE * ef_vals + 0.5, 0, 1))[:, :3]

            launch_viewer(o3d_mesh(all_output_vertices, faces, ef_rgb),
                          window_name=f'Surface — {SURF_NAME} — {ef_name}')
            launch_viewer(o3d_mesh(vertices, faces, ef_rgb),
                          window_name=f'Sphere  — {SURF_NAME} — {ef_name}')
            print(f'  {ef_name}: range [{ef_vals.min():.4f}, {ef_vals.max():.4f}]')
else:
    print(f'Eigenfunction viewer inactive (VIS_MODE={VIS_MODE!r}).'
          ' Set VIS_MODE="neural_eigenfunctions" and re-run from Step 3.')

Eigenfunction viewer inactive (VIS_MODE='normals'). Set VIS_MODE="neural_eigenfunctions" and re-run from Step 3.
